# Tensorflow lite model

Converting the Keras model to lite model for egde devices.

In [1]:
# importing the libraries
import numpy as np
import os
import h5py
import matplotlib.pyplot as plt
from tensorflow import keras
import tensorflow as tf
from sys import getsizeof


In [2]:
tf.__version__


'2.18.0'

In [3]:
# to get model size
def get_folder_size(folder_path: str):
    size = 0
    for path, dirs, files in os.walk(folder_path):
        for file in files:
            fp = os.path.join(path, file)
            size += os.path.getsize(fp)
    return size


In [4]:
KERAS_MODEL_NAME = '../models/3_sl_aug_model.keras'


In [5]:
model = keras.models.load_model(KERAS_MODEL_NAME)


In [6]:
model.summary()


Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ conv2d (Conv2D)                 │ (None, 28, 28, 16)     │           160 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization             │ (None, 28, 28, 16)     │            64 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d (MaxPooling2D)    │ (None, 14, 14, 16)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_1 (Conv2D)               │ (None, 14, 14, 32)     │         4,640 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 14, 14, 32)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_1           │ (None, 14, 14, 32)     │           128 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_1 (MaxPooling2D)  │ (None, 7, 7, 32)       │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_2 (Conv2D)               │ (None, 7, 7, 64)       │        18,496 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_1 (Dropout)             │ (None, 7, 7, 64)       │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_2           │ (None, 7, 7, 64)       │           256 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_2 (MaxPooling2D)  │ (None, 4, 4, 64)       │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ flatten (Flatten)               │ (None, 1024)           │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 512)            │       524,800 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_2 (Dropout)             │ (None, 512)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 24)             │        12,312 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 1,121,490 (4.28 MB)

 Trainable params: 560,632 (2.14 MB)

 Non-trainable params: 224 (896.00 B)

 Optimizer params: 560,634 (2.14 MB)

In [7]:
print(f'Keras Model Size: {get_folder_size(KERAS_MODEL_NAME)} Bytes')


Keras Model Size: 0 Bytes


In [8]:
# creating test set to calculate model accuracy
import pandas as pd
valid_df = pd.read_csv("../data/asl_data/sign_mnist_valid.csv")

# Separate out our target values
y_test = valid_df['label']
del valid_df['label']

# Separate out our image vectors
x_test = valid_df.values

y_test_numpy = y_test.values
y_test = keras.utils.to_categorical(y_test, 24)

# Normalize our image data
x_test = x_test / 255

x_test = x_test.reshape(-1, 28, 28, 1)


In [9]:
loss, acc = model.evaluate(x_test, y_test)
print(f"loss: {loss}, Accuracy: {acc}")


225/225 ━━━━━━━━━━━━━━━━━━━━ 2s 5ms/step - accuracy: 0.7400 - loss: 0.7474
loss: 0.7518722414970398, Accuracy: 0.7359174489974976


## TF Lite Model

In [10]:
TF_LITE_MODEL_NAME = '../models/5_sl_lite.tflite'


In [11]:
# Model convertor
tf_lite_converter = tf.lite.TFLiteConverter.from_keras_model(model)
tf_lite_model = tf_lite_converter.convert()


INFO:tensorflow:Assets written to: C:\Users\aryan\AppData\Local\Temp\tmp22uyja2u\assets


INFO:tensorflow:Assets written to: C:\Users\aryan\AppData\Local\Temp\tmp22uyja2u\assets


Saved artifact at 'C:\Users\aryan\AppData\Local\Temp\tmp22uyja2u'. The following endpoints are available:

* Endpoint 'serve'
  args_0 (POSITIONAL_ONLY): TensorSpec(shape=(None, 28, 28, 1), dtype=tf.float32, name='input_layer')
Output Type:
  TensorSpec(shape=(None, 24), dtype=tf.float32, name=None)
Captures:
  2054616177488: TensorSpec(shape=(), dtype=tf.resource, name=None)
  2054616179408: TensorSpec(shape=(), dtype=tf.resource, name=None)
  2054616179600: TensorSpec(shape=(), dtype=tf.resource, name=None)
  2054616178064: TensorSpec(shape=(), dtype=tf.resource, name=None)
  2054616178832: TensorSpec(shape=(), dtype=tf.resource, name=None)
  2054616177872: TensorSpec(shape=(), dtype=tf.resource, name=None)
  2054616177680: TensorSpec(shape=(), dtype=tf.resource, name=None)
  2054616179984: TensorSpec(shape=(), dtype=tf.resource, name=None)
  2054616183440: TensorSpec(shape=(), dtype=tf.resource, name=None)
  2054616182672: TensorSpec(shape=(), dtype=tf.resource, name=None)
  2054616

In [12]:
import os

if not os.path.exists(KERAS_MODEL_NAME):
    print(f"Error: The folder '{KERAS_MODEL_NAME}' does not exist!")
else:
    keras_size = get_folder_size(KERAS_MODEL_NAME)
    if keras_size == 0:
        print(f"Error: The folder '{KERAS_MODEL_NAME}' is empty!")
    else:
        ratio = os.path.getsize(TF_LITE_MODEL_NAME) / keras_size
        print(f"Size ratio: {ratio}")


Error: The folder '../models/3_sl_aug_model.keras' is empty!


In [13]:
# Saving the model
with open(TF_LITE_MODEL_NAME, "wb") as f:
    f.write(tf_lite_model)


In [14]:
# size of the saved model
print(f"Model size: {os.path.getsize(TF_LITE_MODEL_NAME)} Bytes")


Model size: 2248356 Bytes


In [15]:
# Size comparison with keras model
# os.path.getsize(TF_LITE_MODEL_NAME) / get_folder_size(KERAS_MODEL_NAME)


## Loading the tflite model

In [16]:
# interpreter for tflite model
interpreter = tf.lite.Interpreter(model_path=TF_LITE_MODEL_NAME)

input_details = interpreter.get_input_details()
output_details = interpreter.get_output_details()


In [17]:
input_details


[{'name': 'serving_default_input_layer:0',
  'index': 0,
  'shape': array([ 1, 28, 28,  1], dtype=int32),
  'shape_signature': array([-1, 28, 28,  1], dtype=int32),
  'dtype': numpy.float32,
  'quantization': (0.0, 0),
  'quantization_parameters': {'scales': array([], dtype=float32),
   'zero_points': array([], dtype=int32),
   'quantized_dimension': 0},
  'sparsity_parameters': {}}]

In [18]:
output_details


[{'name': 'StatefulPartitionedCall_1:0',
  'index': 33,
  'shape': array([ 1, 24], dtype=int32),
  'shape_signature': array([-1, 24], dtype=int32),
  'dtype': numpy.float32,
  'quantization': (0.0, 0),
  'quantization_parameters': {'scales': array([], dtype=float32),
   'zero_points': array([], dtype=int32),
   'quantized_dimension': 0},
  'sparsity_parameters': {}}]

In [19]:
print(f"Input Shape: {input_details[0]['shape']}")
print(f"Input type: {input_details[0]['dtype']}")
print(f"output Shape: {output_details[0]['shape']}")
print(f"output type: {output_details[0]['dtype']}")


Input Shape: [ 1 28 28  1]
Input type: <class 'numpy.float32'>
output Shape: [ 1 24]
output type: <class 'numpy.float32'>


In [20]:
x_test.dtype


dtype('float64')

In [21]:
x_test_numpy = np.array(x_test, dtype=np.float32)
x_test_numpy.dtype


dtype('float32')

In [22]:
x_test_numpy.shape


(7172, 28, 28, 1)

In [23]:
y_test.shape


(7172, 24)

In [24]:
# resize tensor shape
interpreter.resize_tensor_input(input_details[0]['index'], x_test_numpy.shape)
interpreter.resize_tensor_input(output_details[0]['index'], y_test.shape)
interpreter.allocate_tensors()
input_details = interpreter.get_input_details()
output_details = interpreter.get_output_details()


In [25]:
print(f"Input Shape: {input_details[0]['shape']}")
print(f"Input type: {input_details[0]['dtype']}")
print(f"output Shape: {output_details[0]['shape']}")
print(f"output type: {output_details[0]['dtype']}")


Input Shape: [7172   28   28    1]
Input type: <class 'numpy.float32'>
output Shape: [7172   24]
output type: <class 'numpy.float32'>


In [26]:
interpreter.set_tensor(input_details[0]['index'], x_test_numpy)
interpreter.invoke()
tflite_prediction = interpreter.get_tensor(output_details[0]['index'])


In [27]:
print(f"Prediction results shape: {tflite_prediction.shape}")
prediction_classes = np.argmax(tflite_prediction, axis=1)


Prediction results shape: (7172, 24)


In [28]:
prediction_classes


array([ 6,  5, 19, ...,  2,  4,  2])

In [29]:
y_test_numpy


array([6, 5, 9, ..., 2, 4, 2])

In [30]:
from sklearn.metrics import accuracy_score
acc = accuracy_score(prediction_classes, y_test_numpy)
print(f"Accuracy of TFLite Model: {acc}")


Accuracy of TFLite Model: 0.7359174567763525


## TFLite Model Float16

converting to more reduced size model by using float 16 instead of float32 

In [31]:
TF_LITE_FLOAT16_MODEL_NAME = "../models/tf_lite_float16.tflite"


In [32]:
tf_lite_converter = tf.lite.TFLiteConverter.from_keras_model(model)
tf_lite_converter.optimizations = [tf.lite.Optimize.DEFAULT]
tf_lite_converter.target_spec.supported_types = [tf.float16]
tf_lite_model = tf_lite_converter.convert()


INFO:tensorflow:Assets written to: C:\Users\aryan\AppData\Local\Temp\tmprfa5rcsz\assets


INFO:tensorflow:Assets written to: C:\Users\aryan\AppData\Local\Temp\tmprfa5rcsz\assets


Saved artifact at 'C:\Users\aryan\AppData\Local\Temp\tmprfa5rcsz'. The following endpoints are available:

* Endpoint 'serve'
  args_0 (POSITIONAL_ONLY): TensorSpec(shape=(None, 28, 28, 1), dtype=tf.float32, name='input_layer')
Output Type:
  TensorSpec(shape=(None, 24), dtype=tf.float32, name=None)
Captures:
  2054616177488: TensorSpec(shape=(), dtype=tf.resource, name=None)
  2054616179408: TensorSpec(shape=(), dtype=tf.resource, name=None)
  2054616179600: TensorSpec(shape=(), dtype=tf.resource, name=None)
  2054616178064: TensorSpec(shape=(), dtype=tf.resource, name=None)
  2054616178832: TensorSpec(shape=(), dtype=tf.resource, name=None)
  2054616177872: TensorSpec(shape=(), dtype=tf.resource, name=None)
  2054616177680: TensorSpec(shape=(), dtype=tf.resource, name=None)
  2054616179984: TensorSpec(shape=(), dtype=tf.resource, name=None)
  2054616183440: TensorSpec(shape=(), dtype=tf.resource, name=None)
  2054616182672: TensorSpec(shape=(), dtype=tf.resource, name=None)
  2054616

In [33]:
with open(TF_LITE_FLOAT16_MODEL_NAME, "wb") as f:
    f.write(tf_lite_model)


In [34]:
print(f"Model size: {os.path.getsize(TF_LITE_FLOAT16_MODEL_NAME)} Bytes")


Model size: 1128944 Bytes


In [35]:
# size comparison
# os.path.getsize(TF_LITE_FLOAT16_MODEL_NAME) / get_folder_size(KERAS_MODEL_NAME)


In [36]:
os.path.getsize(TF_LITE_FLOAT16_MODEL_NAME) / \
    os.path.getsize(TF_LITE_MODEL_NAME)


0.5021197710682828

In [37]:
interpreter = tf.lite.Interpreter(model_path=TF_LITE_MODEL_NAME)
input_details = interpreter.get_input_details()
output_details = interpreter.get_output_details()
# resize tensor shape
interpreter.resize_tensor_input(input_details[0]['index'], x_test_numpy.shape)
interpreter.resize_tensor_input(output_details[0]['index'], y_test.shape)
interpreter.allocate_tensors()
input_details = interpreter.get_input_details()
output_details = interpreter.get_output_details()

interpreter.set_tensor(input_details[0]['index'], x_test_numpy)
interpreter.invoke()
tflite_prediction = interpreter.get_tensor(output_details[0]['index'])
prediction_classes = np.argmax(tflite_prediction, axis=1)

acc = accuracy_score(prediction_classes, y_test_numpy)
print(f"Accuracy of TFLite Model: {acc}")


Accuracy of TFLite Model: 0.7359174567763525
